# The micro tier — the throughput scan

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/micro_scan.ipynb)

The micro tier is the nano body widened and deepened by one rule (every address scaled with the width; `bodies.SIZES`): three sizes above Pollux dual's 9.3 million parameters, Pollux's dual-hub layout kept. Before any of them trains for a week, this notebook prices them on the card: the model step at three batch shapes on random bytes, then the real trainer on the fineweb-edu stream for 300 steps at the picked shape, timed from a warm step (the data plane in its child process, the clip, both optimizers, bf16 autocast, eager: the pace a long run holds). Out come bytes a second, the card's memory, the bytes a day and a week one notebook would train, the bytes a parameter a week makes, and the hours the 2s diet (16.1 billion bytes) would take at each size. Nothing trained here is kept; the record ships to the repo's `preflight/` folder.

| craft | shape | parameters at context 4,096 |
|---|---|---|
| nano-pollux-dual-v1.1 (the reference row) | d256, 6 blocks, 8 heads, dual hubs in blocks 0 2 4 at K32 D16 | 9,267,281 |
| micro26-pollux-dual-v1.1 | d384, 8 blocks, 6 heads, dual hubs in blocks 0 2 4 6 at K48 D24 | 25,787,841 |
| micro45-pollux-dual-v1.1 | d512, 8 blocks, 8 heads, dual hubs in blocks 0 2 4 6 at K64 D32 | 44,944,065 |
| micro66-pollux-dual-v1.1 | d512, 12 blocks, 8 heads, dual hubs in blocks 0 2 4 6 8 10 at K64 D32 | 66,077,601 |

Length: about 45 minutes on an RTX PRO 6000 (the nano row a few minutes, the 66 M row about fifteen, the benches and the stream's start between). The nano runs first, so the reference pace (610 kB/s on 2026-10-19 on that card) is read back before the bigger bodies. Nothing here needs a decision; the week's run does (its plan is in the research record).


In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART RUNTIME if the version line below changes.
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.57"
import torch
import geolip.nano_beatrix as N
import geolip.alephllm as A
from geolip.nano_beatrix import nano_v1 as V1
from geolip.nano_beatrix import micro as M
print('geolip.nano_beatrix', N.__version__, '| geolip.alephllm', A.__version__, '| torch', torch.__version__,
      '| card', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 57), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart, then rerun from C1')
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()                                   # the value is never printed; None = the record stays local
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN

# ============ THE DECISION BLOCK ============
OUT = '/content/micro_scan'                           # the session folder: the births, the scan's runs (removed after each), the record
REPO = V1.REPO                                        # AbstractPhil/alephllm-nano-castor-pollux: the record ships to its preflight/ folder
NAMES = list(M.SCAN_NAMES)                            # the nano reference row and the three micro sizes, in that order; drop a name to skip it
SHAPES = M.SHAPES                                     # micro-batch x accumulation at the 2s step (262,144 bytes): 16 x 4 (the 2s shape), 8 x 8, 32 x 2
STEPS, WARM = 300, 30                                 # the stream run's steps, and the warm steps left outside the clock
SHIP = True                                           # the record to the repo (the weights never); False keeps it in OUT
print('decision block:', len(NAMES), 'crafts:', ', '.join(NAMES), '| shapes', list(SHAPES), '| steps', STEPS, 'warm', WARM,
      '| ships to', REPO if SHIP and HF_TOKEN else 'nowhere (local record)')


In [ ]:
# C2 — THE SCAN: for each craft its parameter count, the bench at every shape, the picked shape's stream run, the projection;
# the table printed after every craft; the record at OUT/micro_scan.json and, when SHIP, in the repo's preflight/ folder.
import time
t0 = time.time()
RES = M.scan(OUT, NAMES, SHAPES, STEPS, WARM, hf_token=HF_TOKEN if SHIP else None)
print(f'{len(RES["crafts"])} crafts in {(time.time() - t0) / 60:.0f} minutes on {RES["card"]}')
print(M.scan_table(RES))


In [ ]:
# C3 — THE READING: the week's bytes at one and two notebooks per size, the bytes a parameter a week makes, the hours of the 2s
# diet and of the 4x diet; the shipped record checked in the repo.
from huggingface_hub import HfApi
print(M.scan_table(RES))
print()
print('| craft | B a week, one notebook | two notebooks | bytes a parameter a week (one) | the 2s diet | 4x |')
print('|---|---|---|---|---|---|')
for c in RES['crafts']:
    print(f"| {c['name']} | {c['week_B']:.0f} | {2 * c['week_B']:.0f} | {c['bytes_per_param_week']:,.0f} | {c['diet_hours']:.1f} h | {c['diet4x_hours']:.1f} h |")
if RES.get('shipped'):
    files = set(HfApi(token=HF_TOKEN).list_repo_files(REPO))
    print('shipped', RES['shipped'], 'READ BACK OK' if RES['shipped'] in files else 'NOT VISIBLE after upload')
else:
    print('the record stayed local:', f'{OUT}/micro_scan.json')


## How to change the run

- **Fewer or other bodies**: edit `NAMES` in C1 (any craft in `V1.ALL`; the micro crafts are `micro26 / micro45 / micro66-pollux-dual-v1.1`, the nano bodies `nano-<body>-v1.1`).
- **Other batch shapes**: `SHAPES` in C1; the first is the preferred shape, another is taken only when it is 10 percent faster and fits the card (`nano_v1.pick_shape`).
- **A longer stream run**: `STEPS` and `WARM` in C1 (300 and 30 take a few minutes a body on an RTX PRO 6000).
- **No ship**: `SHIP = False`; the record stays at `OUT/micro_scan.json`.
- The micro crafts' registered plan is the 2s diet; the schooled mixture and the week's run come with their own notebook once the size is chosen.
